In [1]:
%pip install -q requests torch bitsandbytes transformers sentencepiece accelerate sentence-transformers huggingface-hub kagglehub openai kagglehub[pandas-datasets]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: C:\Users\ricca\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [12]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: C:\Users\ricca\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [39]:
import pandas as pd
import os
import kagglehub
import torch
from openai import OpenAI
import json
from kagglehub import KaggleDatasetAdapter
from dotenv import load_dotenv

Devo caricare un tot di dataset possibilmente di argomenti diversi

In [40]:
datasets = [
    "akshaydattatraykhare/diabetes-dataset",  # 92MB unzipped
    "mathchi/diabetes-data-set",  # 85MB unzipped
    "jillanisofttech/brain-stroke-dataset",  # 78MB unzipped
    "jillanisofttech/brain-tumor",  # 2.1MB unzipped
    "naddamuhhamed/sleepy-driver-eeg-brainwave-data",  # 3.2MB unzipped
    "utkarshx27/non-alcohol-fatty-liver-disease",  # 4.5MB unzipped
    "joebeachcapital/cirrhosis-patient-survival-prediction",  # 1.8MB unzipped
    "sayeemmohammed/anemia-detection",  # 2.3MB unzipped
    "imtkaggleteam/tuberculosis",  # 5.6MB unzipped
    "cid007/mental-disorder-classification",  # 88MB unzipped
    "parvezalmuqtadir2348/postpartum-depression",  # 67MB unzipped
    "arashnic/the-depression-dataset",  # 3.4MB unzipped
    "yasserh/horse-survival-dataset",  # 4.7MB unzipped
    "andrewmvd/fetal-health-classification",  # 98MB unzipped
    "citizen-ds-ghana/health-facilities-gh",  # 1.2MB unzipped
    "steveahn/memory-test-on-drugged-islanders-data",  # 95MB unzipped
    "vuppalaadithyasairam/kidney-stone-prediction-based-on-urine-analysis",  # 82MB unzipped
    "joebeachcapital/differentiated-thyroid-cancer-recurrence",  # 2.8MB unzipped
    "kingburrito666/cannabis-strains",  # 5.1MB unzipped
    "muhammedzidan/egyptian-doctors",  # 4.3MB unzipped
]


In [41]:
file_path = 'datasets/kingburrito666/cannabis-strains/versions/9/cannabis.csv'

df = pd.read_csv(file_path)

In [ ]:
load_dotenv()

1. Domanda: Quali sono gli effetti più comuni riportati dagli utenti per lo strain '100-Og'?
   Risposta: Gli effetti più comuni riportati dagli utenti per lo strain '100-Og' sono Creative, Energetic, Tingly, Euphoric, Relaxed.

2. Domanda: Qual è il rating dello strain '98-White-Widow'?
   Risposta: Il rating dello strain '98-White-Widow' è 4.7.

3. Domanda: Quali sono i sapori associati allo strain '1024'?
   Risposta: I sapori associati allo strain '1024' sono Spicy/Herbal, Sage, Woody.

4. Domanda: Qual è il tipo dello strain con il rating più alto?
   Risposta: Lo strain con il rating più alto è '98-White-Widow' ed è di tipo hybrid.

5. Domanda: Cosa si dice nella descrizione dello strain '1024' riguardo ai suoi effetti?
   Risposta: Nella descrizione dello strain '1024' si menziona che produce effetti uplifted, happy, relaxed, energetic, creative.


In [ ]:
%pip install -q langchain-community
%pip install -q langchain
%pip install -q faiss-cpu
%pip install -q tiktoken

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: C:\Users\ricca\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [25]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain.llms import OpenAI

Conversione del dataset in testo per creare gli embeddings

In [ ]:
text_parts = []

# Aggiungi informazione sulla struttura
text_parts.append(f"Descrizione del dataset con {len(df)} record e colonne: {', '.join(df.columns)}\n\n")

# Aggiungi statistiche di base
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        text_parts.append(f"Statistiche per {col}: min={df[col].min()}, max={df[col].max()}, "
                            f"media={df[col].mean():.2f}, mediana={df[col].median():.2f}\n")

# Aggiungi i dati effettivi
text_parts.append("\nContenuto del dataset:\n")

# Limita a un numero ragionevole di righe se il dataset è grande, serve davvero?
sample_size = min(5000, len(df))
sample_df = df.sample(n=sample_size) if len(df) > sample_size else df

for idx, row in sample_df.iterrows():
    row_text = f"Record {idx}: " + " | ".join([f"{col}={row[col]}" for col in df.columns])
    text_parts.append(row_text + "\n")

text_parts = "\n".join(text_parts)

Creazione embeddings e vector store

In [44]:
# Suddividi il testo in chunk, qui dobbiamo valutare la strategia di chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200
)
chunks = text_splitter.split_text(text_parts)
print(f"Dataset suddiviso in {len(chunks)} chunks.")

# Crea embeddings e store vettoriale
vector_store = FAISS.from_texts(chunks, OpenAIEmbeddings())
print("Vector store creato con successo.")

Dataset suddiviso in 1185 chunks.
Vector store creato con successo.


Generazione di un riassunto riguardante il dataset, bisogna valutare se tenerlo o meno

In [46]:
query = "Genera un riassunto del dataset che rappresenti i temi principali contenuti in questo dataset"

# Recupera i chunk più rilevanti
docs = vector_store.similarity_search(query, k=10)

# Unisci i contenuti
summary = "\n\n".join([doc.page_content for doc in docs])

Qui generiamo le domande

In [37]:
# Prompt per generare domande
question_template = """
Tu sei un esperto nell'analisi di dati. Basandoti sul seguente contenuto del dataset, 
genera {num_questions} domande aperte significative che esplorino gli aspetti chiave dei dati.
Le domande dovrebbero essere variate, coprire diversi aspetti del dataset e richiedere una
comprensione approfondita dei dati.

Contenuto del dataset:
{dataset_context}

Genera solo le domande, una per riga.
"""

question_prompt = PromptTemplate(
    input_variables=["num_questions", "dataset_context"],
    template=question_template
)

# Genera le domande
question_chain = LLMChain(llm=OpenAI(temperature=0.7), prompt=question_prompt)
questions_text = question_chain.run(num_questions=5, dataset_context=summary)

# Elabora il risultato
questions = [q.strip() for q in questions_text.split('\n') if q.strip()]

print(questions)

['1. Quali sono gli effetti più comuni riportati dagli utenti per ogni tipo di cannabis presente nel dataset?', '2. Cosa distingue le varietà di cannabis che hanno ottenuto un punteggio di 4.5 o superiore rispetto a quelle con un punteggio inferiore?', '3. Quali sono le caratteristiche più comuni delle descrizioni delle varietà di cannabis che hanno ottenuto un punteggio di 5.0?', '4. Come si differenziano le varietà di cannabis hybride dalle varietà di cannabis indica e sativa in termini di effetti, sapore e descrizione?', '5. Quali sono le varietà di cannabis con un punteggio di 4.5 o superiore che sono state create da ibridi di altre varietà di cannabis?']


In [38]:
"""Genera risposte alle domande utilizzando il retrieval."""
results = []

answer_template = """
Basandoti sul seguente contesto estratto dal dataset, rispondi alla domanda in modo dettagliato e approfondito.

Contesto:
{context}

Domanda: {question}

Risposta:
"""

answer_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=answer_template
)

answer_chain = LLMChain(llm=OpenAI(temperature=0.7), prompt=answer_prompt)

for q in questions:
    # Recupera il contesto rilevante
    docs = vector_store.similarity_search(q, k=3)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    # Genera la risposta
    answer = answer_chain.run(context=context, question=q)
    
    results.append({
        "domanda": q,
        "risposta": answer.strip()
    })

print(results)

[{'domanda': '1. Quali sono gli effetti più comuni riportati dagli utenti per ogni tipo di cannabis presente nel dataset?', 'risposta': "Il dataset fornisce informazioni riguardo ai diversi tipi di cannabis presenti, in particolare per ogni record sono elencati la varietà (Strain), il tipo (Type), la valutazione (Rating), gli effetti (Effects), il gusto (Flavor) e una descrizione (Description).\n\nPer rispondere alla domanda, è necessario analizzare i dati presenti nella colonna Effects per ogni tipo di cannabis. \n\n1) Hybrid:\n- Strain: 100-Og\n- Effects: Creative, Energetic, Tingly, Euphoric, Relaxed\n\nGli effetti più comuni riportati dagli utenti per questa varietà di cannabis sono:\n- Creatività: questa cannabis sembra avere un effetto stimolante sulla creatività delle persone, che potrebbe essere utile per artisti o persone che stanno cercando di trovare nuove idee.\n- Energia: gli utenti riportano un effetto energetico, che potrebbe essere utile per chi ha bisogno di un po' di 